# Grants.gov Scraper — quickstart

Author: **Public Records**.

This notebook runs on [Google Colab](https://colab.research.google.com/) and [Binder](https://mybinder.org/) with no extra account setup.
It shows a small public snapshot first (when one exists), then optionally runs [`publicrecords/grants-gov-scraper`](https://apify.com/publicrecords/grants-gov-scraper) on **your** Apify account.

The live Actor call is billed to you. Leave the token prompt blank to skip it.
The token is read with `getpass` (never hardcoded, never printed, never read from the environment).

- Apify Store: https://apify.com/publicrecords/grants-gov-scraper
- Examples repo: https://github.com/GeminiGeorge22/publicrecords-grants-gov-scraper-examples
- Public snapshot: https://huggingface.co/datasets/Publicrecords/us-open-federal-grants

## Links

- MCP servers: https://github.com/GeminiGeorge22/publicrecords-mcp
- Agent Skills: https://github.com/GeminiGeorge22/publicrecords-skills
- Beta testers guide: https://github.com/GeminiGeorge22/publicrecords-mcp/blob/main/BETA.md

Runs use **your** Apify account. Never hardcode or print a token. Author: Public Records.


In [1]:
# Works on Colab and Binder. Safe to re-run.
%pip install -q apify-client pandas datasets huggingface_hub pyarrow


Note: you may need to restart the kernel to use updated packages.


## Public snapshot (no token needed)

Loads [`Publicrecords/us-open-federal-grants`](https://huggingface.co/datasets/Publicrecords/us-open-federal-grants) and shows a small table. This does not call Apify.

In [2]:
from datasets import load_dataset
import pandas as pd
from IPython.display import display

DATASET = 'Publicrecords/us-open-federal-grants'
df_public = load_dataset(DATASET, split='train[:25]').to_pandas()
print(f'{DATASET}: {len(df_public)} rows shown, {len(df_public.columns)} columns')
display(df_public)


/workspace/kaggle-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generating train split:   0%|          | 0/787 [00:00<?, ? examples/s]

Generating train split: 100%|██████████| 787/787 [00:00<00:00, 74947.60 examples/s]

Publicrecords/us-open-federal-grants: 25 rows shown, 11 columns


,id,number,title,agencyCode,agency,openDate,closeDate,oppStatus,docType,cfdaList,url
0,356893,PAR-25-144,Dissemination and Implementation Research in H...,HHS-NIH11,National Institutes of Health,10/29/2024,01/07/2028,posted,synopsis,93.279;93.242;93.273;93.866;93.847;93.307;93.1...,https://www.grants.gov/search-results-detail/3...
1,356920,PAR-25-143,Dissemination and Implementation Research in H...,HHS-NIH11,National Institutes of Health,10/31/2024,01/07/2028,posted,synopsis,93.279;93.242;93.273;93.989;93.307;93.172;93.8...,https://www.grants.gov/search-results-detail/3...
2,356907,PAR-25-233,Dissemination and Implementation Research in H...,HHS-NIH11,National Institutes of Health,10/30/2024,01/07/2028,posted,synopsis,93.399;93.242;93.121;93.279;93.310;93.113;93.8...,https://www.grants.gov/search-results-detail/3...
3,357021,PAR-25-221,Population Approaches to Reducing Alcohol-rela...,HHS-NIH11,National Institutes of Health,11/06/2024,01/07/2027,posted,synopsis,93.396;93.273,https://www.grants.gov/search-results-detail/3...
4,357308,PAR-25-286,Neuromodulation/Neurostimulation Device Develo...,HHS-NIH11,National Institutes of Health,11/21/2024,01/07/2028,posted,synopsis,93.242,https://www.grants.gov/search-results-detail/3...
5,357941,PAR-25-101,Cutting-Edge Basic Research Awards (CEBRA) (R2...,HHS-NIH11,National Institutes of Health,12/30/2024,08/11/2027,posted,synopsis,93.279,https://www.grants.gov/search-results-detail/3...
6,329436,DE-FOA-0002265,University Nuclear Leadership Program&ndash; S...,DOE-ID,Idaho Field Office,10/15/2020,10/14/2030,posted,synopsis,81.121,https://www.grants.gov/search-results-detail/3...
7,357568,PAR-25-298,Research Enhancement Award Program (REAP) for ...,HHS-NIH11,National Institutes of Health,12/06/2024,01/07/2028,posted,synopsis,93.839;93.173;93.279;93.840;93.273;93.395;93.8...,https://www.grants.gov/search-results-detail/3...
8,357288,PAR-26-001,"Screening, Brief Intervention and Referral to ...",HHS-NIH11,National Institutes of Health,11/21/2024,05/07/2027,posted,synopsis,93.307;93.273;93.399;93.121;93.279;93.313,https://www.grants.gov/search-results-detail/3...
9,356690,PAR-25-098,Interventions to expand cancer screening and p...,HHS-NIH11,National Institutes of Health,10/10/2024,01/07/2027,posted,synopsis,93.399;93.307;93.121;93.313,https://www.grants.gov/search-results-detail/3...


## Get fresh data (optional)

Enter your own Apify API token when prompted. Leave it blank to skip the live run. The token is not stored and is not printed. Usage is billed to your Apify account.

Get a token at https://console.apify.com/settings/integrations

In [3]:
import getpass
import pandas as pd
from IPython.display import display

# Never hardcode a token. Never print it. Do not read it from the environment.
token = ''
try:
    token = getpass.getpass('Apify API token (leave blank to skip the live Actor run): ')
except Exception:
    token = ''
token = (token or '').strip()

ACTOR = 'publicrecords/grants-gov-scraper'
RUN_INPUT = {'status': 'posted', 'maxItems': 15}

if not token:
    print('No token entered. Skipping the live Actor run. The table above is from the public Hugging Face dataset and did not need a token.')
else:
    from apify_client import ApifyClient
    client = ApifyClient(token)
    token = ''  # drop the local copy before the call returns output
    run = client.actor(ACTOR).call(run_input=RUN_INPUT, timeout_secs=180)
    items = list(client.dataset(run['defaultDatasetId']).iterate_items())
    df_fresh = pd.DataFrame(items)
    display(df_fresh.head(15))
    print('fresh rows', len(df_fresh))


No token entered. Skipping the live Actor run. The table above is from the public Hugging Face dataset and did not need a token.
